# Daemon?
Unsure if this is the correct use but it essentially serves the same purpose.
On program startup, this will monitor all services health and restart them if they fail.

In [2]:
import asyncio
import numpy as np

In [6]:
async def fake_task(crash_time=3):
    await asyncio.sleep(crash_time)
    raise Exception("Crash!")

async def main():
    """Run num_tasks tasks concurrently, 
    check the status of each task every poll_interval seconds, 
    and exit after exit_after seconds."""

    exit_after = 12
    poll_interval = 3
    active_tasks = {}
    num_tasks = 12

    ABC = "abcdefghijklmnopqrstuvwxyz"
    rng = lambda m,M: np.random.randint(m, M)
    # ("fe", fake_task),
    # ("be", fake_task),
    # ("gw", fake_task)
    tasks = [
        ("".join([ABC[rng(0, 26)] for _ in range(3)]) , fake_task)
        for _ in range(num_tasks)
    ]

    while exit_after > 0 or len(active_tasks) > 0:
        try:
            for label, func in tasks:
                lifespan = np.random.randint(1, 7)
                # Second condition is for when daemon is killed
                if label not in active_tasks and exit_after > 0:
                    print(f"Starting task {label} with lifespan {lifespan}")
                    active_tasks[label] = asyncio.create_task(func(lifespan))
                elif active_tasks[label].done():
                    if active_tasks[label].cancelled():
                        print(f"Task {label} was cancelled")
                    elif active_tasks[label].exception():
                        print(f"Task {label} failed with exception: {active_tasks[label].exception()}")
                    else:
                        print(f"Task {label} completed successfully")
                    del active_tasks[label]

        except Exception as e:
            print(f"Caught exception: {e}")
            break

        await asyncio.sleep(poll_interval)
        exit_after -= poll_interval
    
    await asyncio.gather(*active_tasks.values(), return_exceptions=True)
    print("Exiting...")

await main()

Starting task zss with lifespan 3
Starting task soe with lifespan 3
Starting task fdn with lifespan 3
Starting task von with lifespan 2
Starting task nhw with lifespan 4
Starting task rqs with lifespan 2
Starting task iim with lifespan 6
Starting task srp with lifespan 6
Starting task upe with lifespan 6
Starting task ngo with lifespan 4
Starting task gvu with lifespan 4
Starting task puw with lifespan 1
Task soe failed with exception: Crash!
Task von failed with exception: Crash!
Task rqs failed with exception: Crash!
Task puw failed with exception: Crash!
Task zss failed with exception: Crash!
Starting task soe with lifespan 2
Task fdn failed with exception: Crash!
Starting task von with lifespan 2
Task nhw failed with exception: Crash!
Starting task rqs with lifespan 4
Task ngo failed with exception: Crash!
Task gvu failed with exception: Crash!
Starting task puw with lifespan 3
Starting task zss with lifespan 1
Task soe failed with exception: Crash!
Starting task fdn with lifespan 